In [ ]:
import pandas as pd
import os
from paths import (
    raw_dir, 
    processed_dir
)


In [ ]:
splits = ["train", "dev", "test"]

dfs = []
for split in splits:
	path = os.path.join(raw_dir, f"stsb-es-{split}.csv",)
	df = pd.read_csv(
		path,
		names=["sentence1", "sentence2", "score"]
	)
	df["split"] = split
	dfs.append(df)

df = pd.concat(dfs, ignore_index=True)


In [ ]:
df["score_norm"] = df["score"] / 5


In [ ]:
display(df)


In [ ]:
df.groupby("split")["score_norm"].describe()


In [ ]:
def is_valid_text(sentence: str):
    if not isinstance(sentence, str):
        return False
    
    sentence = sentence.strip()
    if len(sentence) <= 0:
        return False

    try:
        sentence.encode("cp1252")
    except UnicodeEncodeError:
        return False

    return True


def clean_dataframe(df: pd.DataFrame):
    valid_rows = []

    for idx, row in df.iterrows():
        sentence1 = row["sentence1"]
        sentence2 = row["sentence2"]
        split = row.get("split", "unknown")

        if is_valid_text(sentence1) and is_valid_text(sentence2):
            valid_rows.append(row)
        else:
            print(f"Removing row {idx} (split={split}):")
            print("  sentence1:", sentence1)
            print("  sentence2:", sentence2)

    return pd.DataFrame(valid_rows).reset_index(drop=True)

df = clean_dataframe(df)


In [ ]:
for split, data in df.groupby("split"):
	path = os.path.join(processed_dir, f"stsb-es-{split}.csv")
	data.to_csv(
		path,
		index=False
	)
	